# 49. Fit fractions and interference fractions

**Objectives:**

- Build a 3-component model and read off `DecayModel.fit_fractions`.
- Read off `DecayModel.interference_fractions`, the pairwise interference matrix.
- Understand why fit fractions do not sum to exactly 1 when components interfere.

Run the cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2, and daughter
indices start at zero. This notebook uses only simulated data and fixed parameter values (no
fit is required to illustrate the fractions).

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, RealImag, Resonance,
)

## 1. A three-component model

A rho(770), an f0(980)-like scalar and a non-resonant term, all coupling to the same pair
`(0, 1)`. Parameter values are fixed by hand (no fit) purely to illustrate the fractions at a
known point.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
components = [
    Resonance("rho", (0, 1), RealImag(1.0, 0.0), mass=0.7753, width=0.1491, spin=1),
    Resonance("f0", (0, 1), RealImag(0.6, 0.3), mass=0.980, width=0.060, spin=0),
    NonResonant(RealImag(0.4, -0.2), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=60, normalization_pair=(0, 1),
)
print("Components:", [c.name for c in model.components])

Components: ['rho', 'f0', 'NR']


## 2. `fit_fractions`

`fit_fractions()` returns each component's fractional contribution to the total rate, in
component order, at the current parameter values (or at an explicit `fit_values` mapping).
Fractions are physical (efficiency excluded) unless an `efficiency` is supplied.

In [3]:
fractions = model.fit_fractions()
for name, fraction in zip((c.name for c in model.components), fractions):
    print(f"{name:>6s}: {float(fraction):.4f}")
print(f"sum = {float(sum(fractions)):.4f}")

   rho: 0.6925
    f0: 0.3116
    NR: 0.1385
sum = 1.1426


## 3. `interference_fractions`

`interference_fractions()` returns the pairwise interference contribution between every pair of
components, as a symmetric matrix with a zero diagonal (a component does not "interfere with
itself" in this decomposition -- that self-term is already inside `fit_fractions`).

In [4]:
interference = model.interference_fractions()
names = [c.name for c in model.components]
print("           " + "  ".join(f"{n:>8s}" for n in names))
for name, row in zip(names, interference):
    print(f"{name:>8s}  " + "  ".join(f"{float(v):8.4f}" for v in row))

                rho        f0        NR
     rho    0.0000   -0.0151   -0.0000
      f0   -0.0151    0.0000   -0.1275
      NR   -0.0000   -0.1275    0.0000


## 4. Why the fractions do not sum to 1

Each fit fraction is `integral(|c_i F_i|^2) / integral(|sum_j c_j F_j|^2)`: the diagonal term of
the coherent sum, normalized by the total. The total rate also contains the off-diagonal cross
terms `2*Re(integral(conj(c_i F_i) c_j F_j))) / integral(|sum_j c_j F_j|^2)` for every pair
`i != j` -- exactly what `interference_fractions` reports (each pair counted once per triangle,
so the upper triangle alone accounts for the missing part). Because these amplitudes interfere,
the identity is

```text
sum(fit_fractions) + sum(upper triangle of interference_fractions) = 1
```

not `sum(fit_fractions) = 1` on its own. A sum of fit fractions above 1 means net constructive
interference on average over the Dalitz plot; below 1 means net destructive interference.

In [5]:
total_interference = np.sum(np.triu(np.asarray(interference), k=1))
print(f"sum(fit_fractions)                    = {float(sum(fractions)):.6f}")
print(f"sum(upper triangle of interference)   = {total_interference:.6f}")
print(f"sum(fit_fractions) + sum(interference) = {float(sum(fractions)) + total_interference:.6f}")

sum(fit_fractions)                    = 1.142626
sum(upper triangle of interference)   = -0.142626
sum(fit_fractions) + sum(interference) = 1.000000


## Summary and exercises

1. Turn off the f0-NR coupling by moving the f0 far from the NR term in mass and see the
   interference fractions shrink toward zero.
2. Pass `fit_values={...}` to `fit_fractions`/`interference_fractions` to evaluate them at a
   different parameter point without rebuilding the model.
3. After a real fit (see lesson 2), call these two methods on `session.model_with_fitted_values`
   or with `fit_values=session.result_values(result)` to report fractions at the fitted point.

Return to the [course guide](TUTORIALS.md).